In [1]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Dataset
df = pd.read_csv("./data/customer_shopping_data.csv")

df.head()

,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,5/8/2022,Kanyon
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,12/12/2021,Forum Istanbul
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,9/11/2021,Metrocity
3,I173702,C988172,Female,66,Shoes,5,3000.85,Credit Card,16/05/2021,Metropol AVM
4,I337046,C189076,Female,53,Books,4,60.60,Cash,24/10/2021,Kanyon


In [5]:
# Add data column
df['Total_Spend'] = df['quantity'] * df['price']

# Only keep neccessary columns
df = df[['category','payment_method','shopping_mall','Total_Spend']]
df.reset_index(inplace=True, drop=True)
df.head()

,category,payment_method,shopping_mall,Total_Spend
0,Clothing,Credit Card,Kanyon,7502.00
1,Shoes,Debit Card,Forum Istanbul,5401.53
2,Clothing,Cash,Metrocity,300.08
3,Shoes,Credit Card,Metropol AVM,15004.25
4,Books,Cash,Kanyon,242.40


** One-Way ANOVA
$H_0$：$\mu_{\text{Cash}} = \mu_{\text{Credit Card}} = \mu_{\text{Debit Card}}$
$H_1$：At least one paymethod has different average spend

In [8]:

model_1way = ols('Total_Spend ~ C(payment_method)', data=df).fit()
anova_1way = sm.stats.anova_lm(model_1way, typ=2)

pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("==== One-way ANOVA Results ====")
print(anova_1way)

==== One-way ANOVA Results ====
                            sum_sq       df    F  PR(>F)
C(payment_method)       7734653.23     2.00 0.22    0.81
Residual          1773223297361.64 99454.00  NaN     NaN


One-way ANOVA results show a p-value of 0.81, meaning we fail to reject $H_0$. This confirms that payment method has no impact on transaction sizes, as the average spend remains consistent whether customers pay with cash, credit card, or debit card.

A Two-way ANOVA tests three separate sets of hypotheses:
1. Main Effect of Product Category (Factor A)：
$H_0$ (Null Hypothesis): The average Total_Spend is the same across all product categories.
$H_1$ (Alternative Hypothesis): At least one product category has a different average Total_Spend.
2. Main Effect of Shopping Mall (Factor B)
$H_0$ (Null Hypothesis): The average Total_Spend is the same across all shopping malls.
$H_1$ (Alternative Hypothesis): At least one shopping mall has a different average Total_Spend.
3. Interaction Effect (Factor A $\times$ Factor B)
$H_0$ (Null Hypothesis): There is no interaction effect between product category and shopping mall (the effect of the category does not depend on the mall).
$H_1$ (Alternative Hypothesis): There is a significant interaction effect between product category and shopping mall.

In [9]:
# 1. Setup the Two-way ANOVA formula with interaction (*)
anova_formula = 'Total_Spend ~ C(category) * C(shopping_mall)'

# 2. Fit the model
model_2way = ols(anova_formula, data=df).fit()

# 3. Perform Two-way ANOVA (Type II Sum of Squares is standard for unbalanced data)
anova_2way_results = sm.stats.anova_lm(model_2way, typ=2)

# 4. Format the float output to avoid scientific notation
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# 5. Display the ANOVA table
print("==== Two-way ANOVA Results ====")
print(anova_2way_results)

==== Two-way ANOVA Results ====
                                        sum_sq         df          F  PR(>F)
C(category)                  857257572835.1113     7.0000 13302.2194  0.0000
C(shopping_mall)                 50044567.5916     9.0000     0.6040  0.7948
C(category):C(shopping_mall)    988252414.5873    63.0000     1.7039  0.0004
Residual                     914903027899.5613 99377.0000        NaN     NaN


Two-way ANOVA Analysis Results:
1. Product Category ($p < 0.001$): We successfully rejected the null hypothesis. Product category remains the primary statistically significant driver of transaction value.
2. Shopping Mall ($p = 0.7948$): We failed to reject the null hypothesis, indicating that shopping at different malls does not inherently lead to a difference in average basket size.
3. Interaction Effect ($p = 0.0004$): We successfully rejected the null hypothesis. There is a statistically significant interaction effect between category and shopping_mall. This implies that the impact of a product category on spending depends on the specific mall it was purchased in (e.g., certain malls may experience significantly higher Technology or Shoe sales than others).